## Production-Grade Financial RAG with RAGWire (Compare Multiple Companies using Ollama)

Till now, we built simple RAG systems…
But those are not enough for real-world use.

In real applications, you need:
- multiple documents
- metadata filtering
- scalable ingestion

And that’s exactly what we’ll build today using RAGWire — a production-grade RAG framework.

This works because RAGWire understands metadata —
so it can compare Apple vs Google correctly instead of mixing data.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from ragwire import RAGWire, setup_logging
import ragwire
logger = setup_logging(log_level="INFO")

print(ragwire.__version__)

rag = RAGWire("config.yaml")

c:\Users\laxmi\anaconda3\envs\ml\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


1.2.6
2026-03-25 19:43:45,817 - ragwire.core.pipeline - INFO - Loading configuration from config.yaml
2026-03-25 19:43:46,190 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-25 19:43:46,191 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-25 19:43:46,808 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=ollama)
2026-03-25 19:43:47,080 - ragwire.core.pipeline - INFO - Metadata extractor loaded from: finance_metadata.yaml
2026-03-25 19:43:47,081 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=ollama, model=qwen3.5:9b)
2026-03-25 19:43:47,737 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://192.168.1.9:6333
2026-03-25 19:43:47,783 - ragwire.vectorstores.qdrant_store - INFO - Deleted collection: multi-company-finance-rag
2026-03-25 19:43:47,784 - ragwire.core.pipeline - INFO - Deleted existing collection for recreation: multi-company

In [3]:
stats = rag.ingest_directory('../data')

2026-03-25 19:43:48,244 - ragwire.core.pipeline - INFO - Found 3 file(s) in ../data
2026-03-25 19:43:48,244 - ragwire.core.pipeline - INFO - Starting ingestion of 3 documents


Ingesting:   0%|          | 0/3 [00:00<?, ?file/s]

2026-03-25 19:44:54,702 - ragwire.core.pipeline - INFO - Processed ..\data\amazon 10k 2025.pdf: 38 chunks


Ingesting:  33%|███▎      | 1/3 [01:06<02:12, 66.45s/file]

2026-03-25 19:45:48,856 - ragwire.core.pipeline - INFO - Processed ..\data\Apple_10k_2025.pdf: 35 chunks


Ingesting:  67%|██████▋   | 2/3 [02:00<00:59, 59.22s/file]

2026-03-25 19:46:34,786 - ragwire.core.pipeline - INFO - Processed ..\data\GOOG-10-K-2025.pdf: 46 chunks


Ingesting: 100%|██████████| 3/3 [02:46<00:00, 55.51s/file]


2026-03-25 19:46:36,950 - ragwire.core.pipeline - INFO - Ingestion complete: 3/3 documents


In [4]:
stats

{'total': 3,
 'processed': 3,
 'skipped': 0,
 'failed': 0,
 'chunks_created': 119,
 'errors': []}

In [5]:
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

from ragwire import RAGWire, setup_logging

logger = setup_logging(log_level="INFO")

# ------------------------------------------------------------------ #
# 2. Tools
# ------------------------------------------------------------------ #
@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.

    Skip this for purely semantic queries with no metadata intent.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=5, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        meta_parts = [
            f"{k}={str(v)[:100]}"
            for k, v in doc.metadata.items()
            if k != "file_name" and v not in (None, "", [])
        ]
        header = f"[{source}" + (f" | {', '.join(meta_parts)}" if meta_parts else "") + "]"
        chunks.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(chunks)


# ------------------------------------------------------------------ #
# 3. Agent with memory
# ------------------------------------------------------------------ #
model = ChatOllama(model="qwen3.5:27b", base_url="http://localhost:11434")
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful financial document assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering — never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "If no relevant documents are found, say so — do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)

In [6]:

config = {"configurable": {"thread_id": "demo"}}


# ------------------------------------------------------------------ #
# 4. Interactive Q&A loop
# ------------------------------------------------------------------ #
print("\nRAG Agent ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue

    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config=config,
    )
    print(f"\nAgent: {response['messages'][-1].content}\n\n\n")



RAG Agent ready. Type 'quit' to exit.

